In [1]:
import geopandas as gpd
import osmnx as ox
import folium
from shapely.geometry import box, Polygon, Point, LineString, MultiLineString
from pathlib import Path
import yaml
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# get path to project root directory
project_root = Path.cwd().parents[0]

# build path to yaml config file
config_path = project_root/"configs"/"paths.yaml"

In [3]:
# load yaml file into a python dictionary called paths
with open(config_path) as f:
    paths = yaml.safe_load(f)

paths

{'data': {'external': 'data/external',
  'raw': 'data/raw',
  'processed': 'data/processed',
  'results': 'results',
  'imagery': 'data/imagery'},
 'outputs': {'dynamic_maps': 'outputs/maps/dynamic',
  'static_maps': 'output/maps/static',
  'figures': 'output/figures',
  'tables': 'outputs/tables'},
 'imagery': {'nakivale_sample': 'data/imagery/nakivale_sample'}}

In [4]:
# build data directory using paths from yaml config
data_dir = project_root/paths['data']['processed']

In [5]:
# build path to output directory for processed grid data
output_dir = project_root/paths['data']['processed']

# build path to output directory for maps
maps_dir = project_root/paths['outputs']['dynamic_maps']

In [ ]:
roads = gpd.read_file(output_dir/"nakivale_osm_highways_utm36N.geojson")
grid = gpd.read_file(output_dir/"nakivale_1km_grids.geojson")


<Projected CRS: EPSG:32636>
Name: WGS 84 / UTM zone 36N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 30°E and 36°E, northern hemisphere between equator and 84°N, onshore and offshore. Belarus. Cyprus. Egypt. Ethiopia. Finland. Israel. Jordan. Kenya. Lebanon. Moldova. Norway. Russian Federation. Saudi Arabia. Sudan. Syria. Türkiye (Turkey). Uganda. Ukraine.
- bounds: (30.0, 0.0, 36.0, 84.0)
Coordinate Operation:
- name: UTM zone 36N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [14]:
grid.head()

,OID,poly_ids,Status,attribution,notes,geometry
0,0,11,todo,,,"POLYGON ((252517.531 -92282.932, 253517.531 -9..."
1,1,11,todo,,,"POLYGON ((252517.531 -91282.932, 253517.531 -9..."
2,2,11,todo,,,"POLYGON ((252517.531 -90282.932, 253517.531 -9..."
3,3,11,todo,,,"POLYGON ((252517.531 -89282.932, 253517.531 -8..."
4,4,11,todo,,,"POLYGON ((252517.531 -88282.932, 253517.531 -8..."


In [12]:
intersections = gpd.overlay(roads, grid, how="intersection")

intersections

,element,id,crossing,highway,name,tactile_paving,crossing:island,traffic_sign,place,date,...,path,wetland,highway:description,passing_places,OID,poly_ids,Status,attribution,notes,geometry
0,way,587251956,None,path,None,None,None,None,None,None,...,None,None,None,None,105,11,todo,,,"LINESTRING (260673.194 -98978.305, 260668.314 ..."
1,way,587252031,None,track,None,None,None,None,None,None,...,None,None,None,None,88,11,todo,,,"LINESTRING (260443.65 -98913.988, 260452.622 -..."
2,way,587251948,None,path,None,None,None,None,None,None,...,None,None,None,None,105,11,todo,,,"LINESTRING (260635.799 -99042.901, 260632.522 ..."
3,way,587251950,None,path,None,None,None,None,None,None,...,None,None,None,None,105,11,todo,,,"LINESTRING (260799.851 -98819.928, 260794.949 ..."
4,way,586670191,None,unclassified,None,None,None,None,None,None,...,None,None,None,None,140,11,todo,,,"LINESTRING (262737.192 -99282.932, 262753.667 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3641,way,848010978,None,track,None,None,None,None,None,None,...,None,None,None,None,473,11,todo,,,"LINESTRING (277516.315 -79996.566, 277517.531 ..."
3642,way,848010978,None,track,None,None,None,None,None,None,...,None,None,None,None,494,11,todo,,,"LINESTRING (277826.931 -80282.932, 277831.754 ..."
3643,way,848010978,None,track,None,None,None,None,None,None,...,None,None,None,None,495,11,todo,,,"LINESTRING (277517.531 -80001.584, 277523.086 ..."
3644,way,851442116,None,unclassified,None,None,None,None,None,None,...,None,None,None,None,473,11,todo,,,"LINESTRING (277517.531 -79996.135, 277516.315 ..."


In [34]:
import pandas as pd

pd.set_option("display.max_rows", 12)

road_counts = (
    intersections[["OID", "id"]]
    .drop_duplicates()
    .groupby("OID")
    .size()
    .reset_index(name="road_segment_count")
)

road_counts.sort_values(by = "road_segment_count" , ascending = False)

,OID,road_segment_count
48,52,154
255,329,34
275,351,33
276,352,32
97,105,29
...,...,...
62,66,1
25,29,1
27,31,1
5,6,1


In [36]:
selected_ids = [351, 127, 77, 112, 449, 380, 225, 625, 53, 108, 103, 444]
sample_grid = grid[grid["OID"].isin(selected_ids)]

# Create complexity column
sample_grid["complexity"] = None

sample_grid.loc[sample_grid["OID"].isin([351, 127, 77, 112]), "complexity"] = "high"
sample_grid.loc[sample_grid["OID"].isin([449, 380, 225, 625]), "complexity"] = "low"
sample_grid.loc[sample_grid["OID"].isin([53, 108, 103, 444]), "complexity"] = "medium"

sample_grid

c:\Users\Zachary\anaconda3\envs\uganda-osm\Lib\site-packages\geopandas\geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,OID,poly_ids,Status,attribution,notes,geometry,complexity
53,53,11,todo,,,"POLYGON ((256517.531 -87282.932, 257517.531 -8...",medium
77,77,11,todo,,,"POLYGON ((258517.531 -94282.932, 259517.531 -9...",high
103,103,11,todo,,,"POLYGON ((259517.531 -84282.932, 260517.531 -8...",medium
108,108,11,todo,,,"POLYGON ((260517.531 -96282.932, 261517.531 -9...",medium
112,112,11,todo,,,"POLYGON ((260517.531 -92282.932, 261517.531 -9...",high
127,127,11,todo,,,"POLYGON ((261517.531 -95282.932, 262517.531 -9...",high
225,225,11,todo,,,"POLYGON ((266517.531 -99282.932, 267517.531 -9...",low
351,351,11,todo,,,"POLYGON ((271517.531 -87282.932, 272517.531 -8...",high
380,380,11,todo,,,"POLYGON ((272517.531 -81282.932, 273517.531 -8...",low
444,444,11,todo,,,"POLYGON ((275517.531 -86282.932, 276517.531 -8...",medium


In [39]:
sample_grid.to_file(
    output_dir/"nakivale_1km_grids_sample.geojson",
    driver="GeoJSON"
)